In [54]:
from langgraph.graph import StateGraph, START, END
from openai import OpenAI
from typing import TypedDict
from dotenv import load_dotenv
import os

In [55]:
load_dotenv()

True

In [56]:
model = OpenAI(
    api_key=os.environ.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)

In [57]:
class LLMState(TypedDict):
    question: str
    answer: str

In [58]:
def llm_qa(state: LLMState) -> LLMState:
    # Extract the question from the state
    question = state["question"]

    # Form a prompt
    prompt = f"You are a helpful assistant. Please answer the following question: {question}"

    # Ask the question to the LLM
    response = model.chat.completions.create(
        messages=[{"role": "user", "content": prompt}],
        model="openai/gpt-oss-120b",
    ).choices[0].message.content

    # Update the state with the answer
    state["answer"] = response

    return state

In [59]:
graph = StateGraph(LLMState)

In [60]:
graph.add_node("llm_qa", llm_qa)

In [61]:
graph.add_edge(START, "llm_qa")
graph.add_edge("llm_qa", END)

In [62]:
workflow = graph.compile()

In [63]:
initial_state = {"question": "How far is the moon from Earth?"}

output_state = workflow.invoke(initial_state)

print(output_state)

{'question': 'How far is the moon from Earth?', 'answer': 'The Moon is roughly **384\u202f000\u202fkilometers (about 239\u202f000\u202fmiles)** away from Earth on average.  \n\nBecause the Moon’s orbit is slightly elliptical, the actual distance changes over the course of each month:\n\n| Position | Approximate distance |\n|----------|----------------------|\n| **Perigee** (closest point) | ~\u202f363\u202f300\u202fkm (≈\u202f225\u202f600\u202fmi) |\n| **Apogee** (farthest point) | ~\u202f405\u202f500\u202fkm (≈\u202f252\u202f000\u202fmi) |\n| **Mean (average) distance** | ~\u202f384\u202f400\u202fkm (≈\u202f238\u202f900\u202fmi) |\n\n**Other useful facts**\n\n- Light (or radio signals) takes about **1.28 seconds** to travel from Earth to the Moon.\n- The Moon completes an orbit around Earth roughly every **27.3 days** (sidereal period) and the same side faces Earth because its rotation period matches this orbital period (tidal locking).\n\nSo, while the exact number varies, you can sa